# Build a dbt Project with Cortex Code

## Using AI to scaffold a dbt project from scratch

In this section, you'll use **Cortex Code** (the AI assistant in Snowsight Workspaces) to generate an entire dbt project from natural language prompts.

**What you'll learn:**
- How to scaffold a dbt project using AI
- Effective prompts for generating dbt models
- The dbt project structure on Snowflake

**How this works:**
1. Create a blank Workspace in Snowsight
2. Open Cortex Code (AI assistant) in the Workspace
3. Use the prompts below to generate each file
4. Compile and run the project

**Prerequisites:** Setup SQL must have been run (creates RAW tables, DBT_SILVER/DBT_GOLD schemas, and the API integration).

---

## Step 1: Create a Blank Workspace

1. In Snowsight, go to **Projects** → **Workspaces**
2. Click **Create Workspace** → **New workspace**
3. Name it `bb_dbt_project`
4. Click **Create**

You now have an empty workspace with a file editor and Cortex Code available.

---

## Step 2: Scaffold the dbt Project

Open Cortex Code in the workspace and use this prompt:

---

### Prompt 1: Project Scaffold

```
I want to set up a dbt project called bb_training_pipeline that connects to my BB_TRAINING database on Snowflake. I'm using the BB_TRAINING_WH warehouse and ACCOUNTADMIN role. 

I need two output layers - a silver layer that writes to the DBT_SILVER schema and a gold layer that writes to the DBT_GOLD schema. Make sure dbt writes to those exact schema names without any prefix.

Set up the full project structure with the config files and folders I'll need.
```

---

## Step 3: Define Sources

Use this prompt to tell dbt about our raw tables:

---

### Prompt 2: Source Definitions

```
I have 4 raw source tables in BB_TRAINING.RAW that I want to transform:
- RAW_CUSTOMERS - business customer profiles (has customer_id, business_name, ABN, industry, state, annual_revenue, employees, years_in_business)
- RAW_LOAN_APPLICATIONS - loan applications from 2021-2024 (has application_id, customer_id, loan_amount, loan_product, status, decision_date)
- RAW_TRANSACTIONS - repayment and fee transactions (has transaction_id, loan_id, customer_id, amount, transaction_type, status)
- RAW_RISK_ASSESSMENTS - credit risk scores per application (has assessment_id, application_id, credit_score, risk_tier)

Create a sources.yml file so I can reference these in my models.
```

---

## Step 4: Create Silver Models

Use this prompt to generate the cleaning/enrichment layer:

---

### Prompt 3: Silver Layer Models

```
Create my silver layer models. These should clean and enrich the raw data:

1. silver_customers - Deduplicate customers (some have multiple records), remove any with invalid ABNs, standardize the state field, and add useful classifications for revenue size and employee count.

2. silver_loan_applications - Enrich loan applications by joining in the risk assessment data (credit scores, risk tier). Remove withdrawn applications since they're not useful. Add some calculated fields like how long it took to make a decision.

3. silver_transactions - Clean up transactions by removing failed ones and invalid amounts. Flag which ones are late payments and categorize them by size.
```

---

## Step 5: Create Gold Models

Use this prompt to generate the business-ready aggregation layer:

---

### Prompt 4: Gold Layer Models

```
Now create my gold layer models. These should aggregate the silver data into business-ready tables:

1. gold_loan_performance - I want a per-loan view showing how each approved loan is performing. Summarize the transaction activity (total repaid, late fees, payment counts) and calculate a repayment ratio and a payment health score from 0-100 where late payments reduce the score.

2. gold_customer_360 - A single row per customer combining their profile from silver_customers with aggregated loan metrics (total loans, total exposure, average credit score, payment health). Classify each customer into a risk category based on their payment health.

3. gold_portfolio_summary - Monthly aggregates of the portfolio broken down by industry, state, and loan product. Show application counts, approval counts, average loan sizes, and approval rates. This is for executive reporting.
```

---

## Step 6: Compile, View DAG, and Run

Once Cortex Code has generated all your files:

### Compile
1. From the **command dropdown** in the workspace menu bar, select **Compile**
2. Click the **execute button** (play icon)
3. Check the **Output** tab for success

### View the DAG
1. Click the **DAG** tab at the bottom of the workspace
2. You should see:
   - 4 source nodes (RAW tables) on the left
   - 3 silver models in the middle
   - 3 gold models on the right
   - Arrows showing data flow and dependencies

### Run
1. Select **Profile: dev** from the dropdown
2. Select **Run** from the command dropdown
3. Click **execute**
4. Watch the Output tab - all 6 models should show `OK`

---

## Step 7: Verify Results

Run this in a SQL worksheet or SQL file within the workspace:

```sql
SELECT 'DBT_SILVER' AS schema, 'SILVER_CUSTOMERS' AS table_name, COUNT(*) AS rows FROM BB_TRAINING.DBT_SILVER.SILVER_CUSTOMERS
UNION ALL SELECT 'DBT_SILVER', 'SILVER_LOAN_APPLICATIONS', COUNT(*) FROM BB_TRAINING.DBT_SILVER.SILVER_LOAN_APPLICATIONS
UNION ALL SELECT 'DBT_SILVER', 'SILVER_TRANSACTIONS', COUNT(*) FROM BB_TRAINING.DBT_SILVER.SILVER_TRANSACTIONS
UNION ALL SELECT 'DBT_GOLD', 'GOLD_LOAN_PERFORMANCE', COUNT(*) FROM BB_TRAINING.DBT_GOLD.GOLD_LOAN_PERFORMANCE
UNION ALL SELECT 'DBT_GOLD', 'GOLD_CUSTOMER_360', COUNT(*) FROM BB_TRAINING.DBT_GOLD.GOLD_CUSTOMER_360
UNION ALL SELECT 'DBT_GOLD', 'GOLD_PORTFOLIO_SUMMARY', COUNT(*) FROM BB_TRAINING.DBT_GOLD.GOLD_PORTFOLIO_SUMMARY
ORDER BY schema, table_name;
```

Row counts should match your SQL notebook output!

---

## Fallback: Pull from GitHub Instead

If Cortex Code isn't generating correct files, you can create a workspace from the pre-built GitHub repo instead:

1. **Projects** → **Workspaces** → **Create Workspace** → **From Git repository**
2. Repository URL: `https://github.com/sfc-gh-ilukman/bb_training_2026.git`
3. Workspace name: `bb_training_dbt`
4. API Integration: `BB_TRAINING_GIT_INTEGRATION`
5. Check **Public repository** → **Create**

Then select **Run** from the command dropdown to execute the pre-built models.

---

## Summary

You just used AI to build a complete dbt project from scratch:

| What you prompted | What was generated |
|---|---|
| Project scaffold | `dbt_project.yml`, `profiles.yml`, `generate_schema_name` macro |
| Source definitions | `sources.yml` with 4 RAW tables |
| Silver models | 3 cleaning/enrichment models using `{{ source() }}` |
| Gold models | 3 aggregation models using `{{ ref() }}` |

### Key Takeaway
Cortex Code can scaffold entire dbt projects from well-structured prompts. The more specific you are about columns, logic, and naming, the better the output.